# USGS dataretrieval Python Package `get_continuous()` Examples

This notebook provides examples of using the Python dataretrieval package to retrieve continuous (instantaneous, or "unit") values data for a United States Geological Survey (USGS) monitoring location. The dataretrieval package provides a collection of functions to get data from the USGS Water Data API and other online sources of hydrology and water quality data.

### Install the Package

Use the following code to install the package if it doesn't exist already within your Jupyter Python environment.

In [ ]:
!pip install dataretrieval

Load the package so you can use it along with other packages used in this notebook.

In [ ]:
from datetime import date

from IPython.display import display

import dataretrieval.waterdata as waterdata

### Basic Usage

The dataretrieval package has several functions that allow you to retrieve data from the USGS Water Data API. This example uses the `get_continuous()` function to retrieve continuous (instantaneous) streamflow data for a USGS monitoring location. The following arguments are supported:

* **monitoring_location_id** (string or iterable of strings): One or more unique monitoring location identifiers. An ID combines the agency code with the location number, separated by a hyphen (e.g. `USGS-10109000`).
* **parameter_code** (string or iterable of strings): One or more 5-digit USGS parameter codes identifying the constituent measured and its units (e.g. `00060` for discharge).
* **time** (string): The date or time interval for which to retrieve observations, given as an RFC 3339 date-time, a bounded or half-bounded interval (e.g. `2021-09-01/2021-09-30`), or an ISO 8601 duration. Continuous data are requested with `time`, and no more than three years of data may be requested per call. If `time` is omitted, the service returns the most recent year of measurements.

#### Example 1: Get unit value data for a specific parameter at a USGS monitoring location between a begin and end date

In [ ]:
# Set the parameters needed for the web service call
siteID = "USGS-10109000"  # LOGAN RIVER ABOVE STATE DAM, NEAR LOGAN, UT
parameterCode = "00060"  # Discharge
startDate = "2021-09-01"
endDate = "2021-09-30"

# Get the data
discharge = waterdata.get_continuous(
    monitoring_location_id=siteID, parameter_code=parameterCode, time=f"{startDate}/{endDate}"
)
print("Retrieved " + str(len(discharge[0])) + " data values.")

### Interpreting the Result

The `get_continuous()` function returns a tuple of two objects: a pandas data frame holding the observed values for the time period requested, and an associated metadata object. The data frame is flat, with a default integer index and one row per observation; the observation timestamps are stored in a tz-aware UTC `time` column rather than being used as the index.

Once you've got the data frame, there are several useful things you can do to explore the data.

In [ ]:
# Display the data frame as a table
display(discharge[0])

Show the data types of the columns in the resulting data frame.

In [ ]:
print(discharge[0].dtypes)

Get summary statistics for the streamflow values.

In [ ]:
discharge[0].describe()

Make a quick time series plot.

In [ ]:
ax = discharge[0].plot(x="time", y="value", style=".")
ax.set_xlabel("Date")
ax.set_ylabel("Streamflow (cfs)")

The other part of the result returned from the `get_continuous()` function is a metadata object that contains information about the query that was executed to return the data. For example, you can access the URL that was assembled to retrieve the requested data from the USGS Water Data API.

In [ ]:
print("The query URL used to retrieve the data from the Water Data API was: " + discharge[1].url)

### Additional Examples

#### Example 2: Get unit values for an individual monitoring location and parameter between a start and end date.

NOTE: By default, start and end date are evaluated as local time, and the result is returned with the timestamps in the local time of the monitoring location.

In [ ]:
site_id = "USGS-05114000"
startDate = "2014-10-10"
endDate = "2014-10-10"

discharge2 = waterdata.get_continuous(
    monitoring_location_id=site_id, parameter_code=parameterCode, time=f"{startDate}/{endDate}"
)
print("Retrieved " + str(len(discharge2[0])) + " data values.")
display(discharge2[0])

#### Example 3: Get unit values for an individual monitoring location for today

In [ ]:
today = str(date.today())
discharge_today = waterdata.get_continuous(
    monitoring_location_id=site_id, parameter_code=parameterCode, time=f"{today}/{today}"
)
print("Retrieved " + str(len(discharge_today[0])) + " data values.")
display(discharge_today[0])

#### Example 4: Retrieve data using UTC times

NOTE: Adding 'Z' to the input time parameters indicates that they are in UTC rather than local time. The time stamps associated with the data returned are still in the local time of the USGS monitoring location.

In [ ]:
discharge_UTC = waterdata.get_continuous(
    monitoring_location_id=site_id,
    parameter_code=parameterCode,
    time="2014-10-10T00:00Z/2014-10-10T23:59Z",
)
print("Retrieved " + str(len(discharge_UTC[0])) + " data values.")
display(discharge_UTC[0])

#### Example 5: Get unit values for two monitoring locations, for a single parameter, between a start and end date

In [ ]:
discharge_multisite = waterdata.get_continuous(
    monitoring_location_id=["USGS-04024430", "USGS-04024000"],
    parameter_code=parameterCode,
    time="2013-10-01/2013-10-01",
)
print("Retrieved " + str(len(discharge_multisite[0])) + " data values.")
display(discharge_multisite[0])

The following example requests the same two-location data as the previous example.

In [ ]:
discharge_multisite = waterdata.get_continuous(
    monitoring_location_id=["USGS-04024430", "USGS-04024000"],
    parameter_code=parameterCode,
    time="2013-10-01/2013-10-01",
    
)
print("Retrieved " + str(len(discharge_multisite[0])) + " data values.")
display(discharge_multisite[0])